# Nested cross-validation baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/02_nested_cv_baseline.ipynb)

**Answers:** Editor comment 1
**Estimated runtime:** 30 min quick / 6 h full · **Hardware:** CPU (T4 helps)
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

The structural core of the revision: 5 outer x 5 repeats of `StratifiedGroupKFold`
grouped on `CNTSCHID`, 5 inner folds, model **and** threshold selected inside the
inner loop, outer test fold touched exactly once. Checkpointed per fold.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from vlpso_xai.data.features import Allowlist, build_design_matrix, single_variable_auc_screen
from vlpso_xai.data.outcome import build_pv_categories
from vlpso_xai.models.registry import get_models
from vlpso_xai.evaluation.nested_cv import run_nested_cv, NestedCVConfig

prim = pd.concat([prim, build_pv_categories(prim)], axis=1)
alw = Allowlist.load(cfg.paths.root / "config" / "predictor_allowlist.yaml")
X_all = build_design_matrix(prim, alw, where="nb02")
print("design matrix:", X_all.shape)

In [ ]:
# --- Empirical leakage screen before anything is fitted ----------------
lab = prim["cat_pv1"]; m = lab.isin(["Low", "High"]).to_numpy()
screen = single_variable_auc_screen(X_all[m], (lab[m] == "High").astype(int).to_numpy(),
                                    allowlist=alw, raise_on_flag=True)
display(screen.head(10))

In [ ]:
# --- Nested CV -----------------------------------------------------------
results = []
for task, (neg, pos) in {"low_vs_high": ("Low","High"),
                         "low_vs_medium": ("Low","Medium"),
                         "medium_vs_high": ("Medium","High")}.items():
    m = prim["cat_pv1"].isin([neg, pos]).to_numpy()
    r = run_nested_cv(
        X_all[m].reset_index(drop=True),
        (prim["cat_pv1"][m] == pos).astype(int).to_numpy(),
        prim.CNTSCHID.to_numpy()[m],
        models=get_models(cfg.section("models", "primary"), fast=QUICK_MODE),
        cfg=NestedCVConfig(
            outer_splits=cfg.section("cv", "outer_splits"),
            outer_repeats=cfg.section("cv", "outer_repeats"),
            inner_splits=cfg.section("cv", "inner_splits"),
            checkpoint_dir=cfg.paths.checkpoints),
        sample_weight=prim.W_FSTUWT.to_numpy()[m], task=task, pv=1)
    results.append(r)
res = pd.concat(results, ignore_index=True)
res.to_parquet(cfg.paths.results / "nested_cv_results.parquet", index=False)
display(res.groupby("task")[["auc", "w_auc", "balanced_accuracy"]].agg(["mean", "std"]))